In [8]:
from pathlib import Path
import h5py

src_dir = Path("/home/flow/code/libero/LIBERO/libero/datasets/libero_10")
hdf5_files = sorted(src_dir.glob("*.hdf5"))

total_files = 0
total_episodes = 0
total_frames = 0

for hdf5_path in hdf5_files:
    total_files += 1
    with h5py.File(hdf5_path, "r") as f:
        demo_names = sorted(f["data"].keys())
        total_episodes += len(demo_names)
        total_frames += sum(f["data"][d]["actions"].shape[0] for d in demo_names)

print("Total files   :", total_files)
print("Total episodes:", total_episodes)
print("Total frames  :", total_frames)

Total files   : 10
Total episodes: 500
Total frames  : 138090


In [ ]:
from pathlib import Path
from lerobot.datasets.lerobot_dataset import HF_LEROBOT_HOME

repo_id = "flow929/ledataset_libero_10"
root = HF_LEROBOT_HOME / repo_id

print("dataset root:", root)
print("exists:", root.exists())

for p in sorted(root.iterdir()):
    print(p.name)

In [ ]:
import h5py

In [ ]:
path = "/home/flow/code/libero/LIBERO/libero/datasets/libero_10/KITCHEN_SCENE4_put_the_black_bowl_in_the_bottom_drawer_of_the_cabinet_and_close_it_demo.hdf5"



In [ ]:
def print_h5_structure(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"[DATASET] {name:60s} shape={obj.shape}, dtype={obj.dtype}")
    else:
        print(f"[GROUP  ] {name}")

with h5py.File(path, "r") as f:
    f.visititems(print_h5_structure)

In [ ]:

with h5py.File(path, "r") as f:
    demo = f["data/demo_0"]

    print("demo keys:", list(demo.keys()))
    print("obs keys :", list(demo["obs"].keys()))

In [ ]:
with h5py.File(path, "r") as f:
    d = f["data/demo_0"]

    print("actions      :", d["actions"].shape, d["actions"].dtype)
    print("rewards      :", d["rewards"].shape, d["rewards"].dtype)
    print("dones        :", d["dones"].shape, d["dones"].dtype)

    for k in d["obs"].keys():
        x = d["obs"][k]
        print(f"obs/{k:15s}: shape={x.shape}, dtype={x.dtype}")

In [ ]:
import numpy as np

with h5py.File(path, "r") as f:
    d = f["data/demo_0"]

    t = 0
    print("action[0] =", d["actions"][t])
    print("reward[0] =", d["rewards"][t])
    print("done[0]   =", d["dones"][t])

    print("joint_states[0] =", d["obs"]["joint_states"][t])
    print("ee_pos[0]       =", d["obs"]["ee_pos"][t])
    print("gripper_states[0] =", d["obs"]["gripper_states"][t])
    

    print("ee_ori[0] =", d["obs"]["ee_ori"][t])
    print("ee_pos[0] =", d["obs"]["ee_pos"][t])
    print("ee_states[0] =", d["obs"]["ee_states"][t])
    # ee_states = ee_pos + ee_ori

In [ ]:
import matplotlib.pyplot as plt

with h5py.File(path, "r") as f:
    imgs = f["data/demo_0/obs/agentview_rgb"]
    hand = f["data/demo_0/obs/eye_in_hand_rgb"]

    print("agentview_rgb:", imgs.shape, imgs.dtype)
    print("eye_in_hand_rgb:", hand.shape, hand.dtype)
    plt.figure(figsize=(8,4))

    plt.subplot(1,2,1)
    plt.imshow(imgs[0])
    plt.title("agentview_rgb[0]")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(hand[0])
    plt.title("eye_in_hand_rgb[0]")
    plt.axis("off")

    plt.show()

In [ ]:
from lerobot.datasets.lerobot_dataset import HF_LEROBOT_HOME
from lerobot.datasets.lerobot_dataset import LeRobotDataset

In [ ]:

with h5py.File(path, "r") as f:
    demo = f["data/demo_0"]

    actions = demo["actions"][:]
    agentview = demo["obs"]["agentview_rgb"]
    wristview = demo["obs"]["eye_in_hand_rgb"][:]

    joint_states = demo["obs"]["joint_states"][:]
    gripper_states = demo["obs"]["gripper_states"][:]
    ee_pos = demo["obs"]["ee_pos"][:]
    ee_ori = demo["obs"]["ee_ori"][:]
    ee_states = demo["obs"]["ee_states"][:]

    print("actions:", actions.shape)
    print("agentview:", agentview.shape)
    print("wristview:", wristview.shape)
    
    print("gripper_states:", gripper_states.shape)

    print("state:", ee_states.shape)

    t = 0
    state = np.concatenate([
        ee_states[t].reshape(-1),
        gripper_states[t].reshape(-1)
    ], axis=0).astype(np.float32)


    print("state:", state[0].shape)

    frame = {
        "observation.images.agentview_rgb": agentview[t],
        "observation.images.eye_in_hand_rgb": wristview[t],
        "observation.state": state,
        "action": actions[t].astype(np.float32),
    }

    print("\nframe keys:", frame.keys())
    print("state shape:", frame["observation.state"].shape)
    print("action shape:", frame["action"].shape)
    print("agent image shape:", frame["observation.images.agentview_rgb"].shape)
    print("wrist image shape:", frame["observation.images.eye_in_hand_rgb"].shape)

In [ ]:
from lerobot.datasets.lerobot_dataset import HF_LEROBOT_HOME
import shutil

our_repo_id = "local/libero_test"
output_path = HF_LEROBOT_HOME / our_repo_id
if output_path.exists():
    shutil.rmtree(output_path)



with h5py.File(path, "r") as f:
    demo = f["data/demo_0"]
    agentview = demo["obs"]["agentview_rgb"][:]
    wristview = demo["obs"]["eye_in_hand_rgb"][:]
    states_6 = demo["obs"]["ee_states"][:]
    states_2 = demo["obs"]["gripper_states"][:]
    print("ee_states :",states_6.shape)
    print("gripper_states :",states_2.shape)
    actions = demo["actions"][:].astype(np.float32)

    state8 = np.concatenate([
        states_6[0].reshape(-1),
        states_2[0].reshape(-1),
    ], axis=0).astype(np.float32)
    print("state8 :",state8.shape)

    dataset = LeRobotDataset.create(
        repo_id=our_repo_id,
        fps=10,
        robot_type="panda",
        features={
            "image": {
                "dtype": "image",
                "shape": (128, 128, 3),
                "names": ["height", "width", "channel"],
            },
            "wrist_image": {
                "dtype": "image",
                "shape": (128, 128, 3),
                "names": ["height", "width", "channel"],
            },
            "state": {
                "dtype": "float32",
                "shape": (8,),
                "names": ["state"],
            },
            "actions": {
                "dtype": "float32",
                "shape": (7,),
                "names": ["actions"],
            },
        },
        image_writer_threads=4,
        image_writer_processes=0,
    )

    time = actions.shape[0]

    for t in range(time):
        state_t = np.concatenate([
            states_6[t].reshape(-1),
            states_2[t].reshape(-1),
        ], axis=0).astype(np.float32)

        frame = {
            "image": agentview[t],
            "wrist_image": wristview[t],
            "state": state_t,
            "actions": actions[t],
            "task": "put the black bowl in the bottom drawer and close it",
        }
        dataset.add_frame(frame)

    dataset.save_episode()


In [ ]:
import inspect
import json
from pathlib import Path
import lerobot
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print("lerobot version:", getattr(lerobot, "__version__", "unknown"))
print("create signature:", inspect.signature(LeRobotDataset.create))
print("save_episode signature:", inspect.signature(LeRobotDataset.save_episode))

root = output_path
print("exists:", root.exists())
print("top-level:", [p.name for p in root.iterdir()] if root.exists() else "no root")

info_path = root / "meta" / "info.json"
print("info exists:", info_path.exists())
if info_path.exists():
    info = json.loads(info_path.read_text())
    print("features:")
    for k, v in info.get("features", {}).items():
        print(k, "->", v.get("dtype"), v.get("shape"))

In [ ]:
from pathlib import Path

for p in sorted(output_path.rglob("*")):
    print(p.relative_to(output_path))
    

In [ ]:
import h5py
import numpy as np
import shutil
from pathlib import Path
from lerobot.datasets.lerobot_dataset import LeRobotDataset, HF_LEROBOT_HOME

path = "/home/flow/code/libero/LIBERO/libero/datasets/libero_10/KITCHEN_SCENE4_put_the_black_bowl_in_the_bottom_drawer_of_the_cabinet_and_close_it_demo.hdf5"

our_repo_id = "local/libero_test"
output_path = HF_LEROBOT_HOME / our_repo_id
if output_path.exists():
    shutil.rmtree(output_path)

with h5py.File(path, "r") as f:
    demo_names = sorted(f["data"].keys())
    print("num demos:", len(demo_names))

    # 先拿第一个 demo 推断 shape
    first_demo = f["data"][demo_names[0]]
    first_agentview = first_demo["obs"]["agentview_rgb"]
    first_wristview = first_demo["obs"]["eye_in_hand_rgb"]
    first_states_6 = first_demo["obs"]["ee_states"]
    first_states_2 = first_demo["obs"]["gripper_states"]
    first_actions = first_demo["actions"]

    state_dim = (
        first_states_6[0].reshape(-1).shape[0]
        + first_states_2[0].reshape(-1).shape[0]
    )
    action_dim = first_actions[0].reshape(-1).shape[0]

    dataset = LeRobotDataset.create(
        repo_id=our_repo_id,
        fps=10,
        robot_type="panda",
        features={
            "image": {
                "dtype": "image",
                "shape": tuple(first_agentview[0].shape),
                "names": ["height", "width", "channel"],
            },
            "wrist_image": {
                "dtype": "image",
                "shape": tuple(first_wristview[0].shape),
                "names": ["height", "width", "channel"],
            },
            "state": {
                "dtype": "float32",
                "shape": (state_dim,),
                "names": ["state"],
            },
            "actions": {
                "dtype": "float32",
                "shape": (action_dim,),
                "names": ["actions"],
            },
        },
        image_writer_threads=4,
        image_writer_processes=0,
    )

    for demo_name in demo_names:
        demo = f["data"][demo_name]

        agentview = demo["obs"]["agentview_rgb"]
        wristview = demo["obs"]["eye_in_hand_rgb"]
        states_6 = demo["obs"]["ee_states"]
        states_2 = demo["obs"]["gripper_states"]
        actions = demo["actions"]

        T = actions.shape[0]
        print(f"processing {demo_name}, length={T}")

        for t in range(T):
            state_t = np.concatenate([
                states_6[t].reshape(-1),
                states_2[t].reshape(-1),
            ], axis=0).astype(np.float32)

            frame = {
                "image": agentview[t],
                "wrist_image": wristview[t],
                "state": state_t,
                "actions": actions[t].astype(np.float32),
                "task": Path(path).stem,
            }
            dataset.add_frame(frame)

        dataset.save_episode(parallel_encoding=False)

print("done")

In [ ]:
import shutil
from pathlib import Path

import cv2
import h5py
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset, HF_LEROBOT_HOME


# ========= 你要改的地方 =========
SRC_DIR = Path("/home/flow/code/libero/LIBERO/libero/datasets/libero_spatial")
REPO_ID = "flow929/le_libero_spatial"
FPS = 10
IMAGE_SIZE = 256
OVERWRITE = True

# 写图片阶段的并发参数
IMAGE_WRITER_PROCESSES = 0
IMAGE_WRITER_THREADS = 8
# ==============================


def resize_rgb(img: np.ndarray, size: int = 256) -> np.ndarray:
    """Resize HWC uint8 RGB image to (size, size, 3)."""
    out = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)
    if out.dtype != np.uint8:
        out = out.astype(np.uint8)
    return out


def build_state(ee_states_t: np.ndarray, gripper_states_t: np.ndarray) -> np.ndarray:
    """Concatenate ee_states and gripper_states into 1D float32 state."""
    state = np.concatenate(
        [
            ee_states_t.reshape(-1),
            gripper_states_t.reshape(-1),
        ],
        axis=0,
    ).astype(np.float32)
    return state


def infer_feature_shapes(first_hdf5: Path):
    """Infer dataset feature shapes from the first demo of the first file."""
    with h5py.File(first_hdf5, "r") as f:
        demo_names = sorted(f["data"].keys())
        if not demo_names:
            raise ValueError(f"No demos found in {first_hdf5}")

        demo = f["data"][demo_names[0]]

        image_shape = (IMAGE_SIZE, IMAGE_SIZE, 3)
        wrist_shape = (IMAGE_SIZE, IMAGE_SIZE, 3)

        state_dim = (
            demo["obs"]["ee_states"][0].reshape(-1).shape[0]
            + demo["obs"]["gripper_states"][0].reshape(-1).shape[0]
        )
        action_dim = demo["actions"][0].reshape(-1).shape[0]

    return image_shape, wrist_shape, state_dim, action_dim


def create_lerobot_dataset(
    repo_id: str,
    image_shape: tuple[int, int, int],
    wrist_shape: tuple[int, int, int],
    state_dim: int,
    action_dim: int,
) -> LeRobotDataset:
    """Create a LeRobot image-based dataset."""
    dataset = LeRobotDataset.create(
        repo_id=repo_id,
        fps=FPS,
        robot_type="panda",
        features={
            "image": {
                "dtype": "image",
                "shape": image_shape,
                "names": ["height", "width", "channel"],
            },
            "wrist_image": {
                "dtype": "image",
                "shape": wrist_shape,
                "names": ["height", "width", "channel"],
            },
            "state": {
                "dtype": "float32",
                "shape": (state_dim,),
                "names": ["state"],
            },
            "actions": {
                "dtype": "float32",
                "shape": (action_dim,),
                "names": ["actions"],
            },
        },
        use_videos=True,
        image_writer_processes=IMAGE_WRITER_PROCESSES,
        image_writer_threads=IMAGE_WRITER_THREADS,
    )
    return dataset


def convert_one_demo(dataset: LeRobotDataset, demo, task_name: str) -> int:
    """
    Convert one demo into one LeRobot episode.
    Returns the number of frames written.
    """
    agentview = demo["obs"]["agentview_rgb"]
    wristview = demo["obs"]["eye_in_hand_rgb"]
    ee_states = demo["obs"]["ee_states"]
    gripper_states = demo["obs"]["gripper_states"]
    actions = demo["actions"]

    T = actions.shape[0]

    # 一些基本一致性检查
    assert agentview.shape[0] == T
    assert wristview.shape[0] == T
    assert ee_states.shape[0] == T
    assert gripper_states.shape[0] == T

    for t in range(T):
        image_t = resize_rgb(agentview[t], IMAGE_SIZE)
        wrist_t = resize_rgb(wristview[t], IMAGE_SIZE)
        state_t = build_state(ee_states[t], gripper_states[t])
        action_t = actions[t].reshape(-1).astype(np.float32)

        frame = {
            "image": image_t,
            "wrist_image": wrist_t,
            "state": state_t,
            "actions": action_t,
            "task": task_name,
        }
        dataset.add_frame(frame)

    dataset.save_episode(parallel_encoding=False)
    return T


def main():
    hdf5_files = sorted(SRC_DIR.glob("*.hdf5"))
    if not hdf5_files:
        raise FileNotFoundError(f"No .hdf5 files found in: {SRC_DIR}")

    output_path = HF_LEROBOT_HOME / REPO_ID
    if OVERWRITE and output_path.exists():
        shutil.rmtree(output_path)

    image_shape, wrist_shape, state_dim, action_dim = infer_feature_shapes(hdf5_files[0])

    dataset = create_lerobot_dataset(
        repo_id=REPO_ID,
        image_shape=image_shape,
        wrist_shape=wrist_shape,
        state_dim=state_dim,
        action_dim=action_dim,
    )

    total_files = 0
    total_episodes = 0
    total_frames = 0

    for hdf5_path in hdf5_files:
        total_files += 1
        print(f"\nProcessing file: {hdf5_path.name}")

        with h5py.File(hdf5_path, "r") as f:
            demo_names = sorted(f["data"].keys())
            print(f"  demos: {len(demo_names)}")

            for demo_name in demo_names:
                demo = f["data"][demo_name]
                task_name = hdf5_path.stem

                num_frames = convert_one_demo(dataset, demo, task_name)

                total_episodes += 1
                total_frames += num_frames
                # print(f"  saved {demo_name}: {num_frames} frames")

    print("\nDone.")
    print(f"Total files    : {total_files}")
    print(f"Total episodes : {total_episodes}")
    print(f"Total frames   : {total_frames}")
    print(f"Output path    : {output_path}")


if __name__ == "__main__":
    main()

In [ ]:
import shutil
from pathlib import Path

import cv2
import h5py
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset, HF_LEROBOT_HOME


# ========= 你要改的地方 =========
SRC_DIR = Path("/home/flow/code/libero/LIBERO/libero/datasets/libero_spatial")
REPO_ID = "flow929/ledataset_libero_spatial"
FPS = 10
IMAGE_SIZE = 256
OVERWRITE = True

# 写图片阶段的并发参数
IMAGE_WRITER_PROCESSES = 0
IMAGE_WRITER_THREADS = 8
# ==============================



def main():
    hdf5_files = sorted(SRC_DIR.glob("*.hdf5"))
    if not hdf5_files:
        raise FileNotFoundError(f"No .hdf5 files found in: {SRC_DIR}")



    total_files = 0
    total_episodes = 0
    total_frames = 0

    for hdf5_path in hdf5_files:
        total_files += 1
        # print(f"\nProcessing file: {hdf5_path.name}")

    with h5py.File(hdf5_path, "r") as f:
        demo_names = sorted(f["data"].keys())

        for demo_name in demo_names:
            demo = f["data"][demo_name]
            total_episodes += 1
            total_frames += demo["actions"].shape[0]
    print("\nDone.")
    print(f"Total files    : {total_files}")
    print(f"Total episodes : {total_episodes}")
    print(f"Total frames   : {total_frames}")
    #print(f"Output path    : {output_path}")


if __name__ == "__main__":
    main()

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

repo_id = "flow929/ledataset_libero_spatial"
ds = LeRobotDataset(repo_id)

print("len(ds) =", len(ds))
print("type(ds) =", type(ds))

episode_ids = set()

for i in range(len(ds)):
    sample = ds[i]
    episode_ids.add(int(sample["episode_index"]))

print("total frames from len(ds):", len(ds))
print("total episodes from episode_index:", len(episode_ids))